# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We'll inspect the available record sets and their structure, always referencing entities by their `@id`.

In [ ]:
# List all record sets by @id
record_sets = metadata.record_sets
print(f"Number of record sets: {len(record_sets)}")
for i, rs in enumerate(record_sets):
    print(f"[{i}] Record set: @id={rs.id}, name={rs.name}")
    print(f"    Description: {rs.description if hasattr(rs, 'description') else ''}")
    # List fields for this record set
    print("    Fields and columns:")
    for field in rs.fields:
        field_id = getattr(field, 'id', None)
        col_id = getattr(field, 'column', None).id if getattr(field, 'column', None) is not None else None
        print(f"        Field: @id={field_id}, name={field.name}, column @id={col_id}")
    print('')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this example, we will use the first record set. Replace with another @id from the overview if needed.
selected_record_set_id = record_sets[0].id

records = list(dataset.records(record_set=selected_record_set_id))
df = pd.DataFrame(records)
print(f"Columns available in record set '@id={selected_record_set_id}':")
print(df.columns.tolist())

# Show the first 5 rows
df.head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll use one numeric field and one grouping field as examples. Update the field `@id`s as appropriate based on your record set fields.

In [ ]:
# Example: Use a numeric field for analysis
# Replace these with actual @id values from your fields/columns
numeric_field_id = None
group_field_id = None

# Try to auto-select a numeric field & group field (for demo)
import numpy as np
for col in df.columns:
    # Attempt to detect a numeric column
    if numeric_field_id is None and np.issubdtype(df[col].dropna().apply(type).mode()[0], (int, float, np.integer, np.floating)):
        numeric_field_id = col
    # Pick any text/categorical field for grouping
    if group_field_id is None and df[col].dtype == object and df[col].nunique() < 20:
        group_field_id = col
    if numeric_field_id and group_field_id:
        break

if numeric_field_id is None or group_field_id is None:
    print('Could not auto-select numeric or group field. Please specify field @id manually.')
else:
    print(f"Numeric field selected: {numeric_field_id}")
    print(f"Group field selected: {group_field_id}")

    # Filtering step
    threshold = df[numeric_field_id].mean()  # use mean as threshold for demonstration
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered {len(filtered_df)} records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df[[numeric_field_id, group_field_id]].head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().rename(columns={numeric_field_id: f"{numeric_field_id}_mean"})
    print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
    print(grouped_df.head())

    # Save for visualization
    vis_grouped_df = grouped_df.copy()

## 5. Visualization

Visualize distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if numeric_field_id is not None and group_field_id is not None:
    # Histogram of numeric field
    df[numeric_field_id].hist(bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Bar plot of means grouped by group_field
    plt.figure(figsize=(8, 4))
    plt.bar(vis_grouped_df[group_field_id], vis_grouped_df[f"{numeric_field_id}_mean"])
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print('Visualization cannot be completed automatically; please set numeric_field_id and group_field_id.')

## 6. Conclusion

In this notebook, we loaded metadata and records from the FAIR^2 mass spectrometry dataset describing proteins isolated from stimulated human mast cells. We explored the available record sets and their fields (by `@id`), loaded a sample into a DataFrame, performed simple EDA including filtering and normalization, grouped by categorical features, and visualized key distributions.

**Key takeaways:**
- The dataset is well-structured with Croissant schema, enabling programmatic loading and referencing by `@id`.
- With `mlcroissant`, you can extract, clean, and visualize any record set using only its schema.

For deeper insights, consult the full schema or dataset documentation for specific protein feature meanings and downstream analysis possibilities!